# Paper QPP results: Table 5 + Figure 3 (offline cold-start dose-response)

Each model family is trained on one workload and evaluated on the blended held-out set
(off-diagonal protocol). The underlying sweeps are produced by
[`01_offline_training_and_testing.ipynb`](01_offline_training_and_testing.ipynb); this
notebook only reloads their results and builds the table + plot.

`results/cold_start_dose_response_offline_qpp_results.csv` merges the three sweeps behind the
paper, with each row's origin in `source_stamp`:

| Rows | Sweep stamp |
|---|---|
| `mean`, `random_forest`, `xgboost`, `lightgbm`, `gp_rbf`, `mlp` (n ≥ 20) | `20260828T082758Z` (full-roster sweep) |
| `mlp` at n = 5/10 | `20260928T173209Z` (MLP-only backfill) |
| `mscn` (unbounded z-scored log-runtime head, the paper's MSCN) | `20260902T141545Z` (MSCN A/B sweep) |
| `mscn_minmax` (original clamped sigmoid head, for comparison only) | `20260828T082758Z` |


## 0 . Setup - load the merged results

All three sweeps share seeds, legs, checkpoints and the fixed held-out split, so their rows
merge directly.


In [ ]:
import json, sys, warnings
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

warnings.filterwarnings("ignore")
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)

RESULTS_DIR = (Path.cwd() / "results").resolve()
assert RESULTS_DIR.is_dir(), RESULTS_DIR
QPP_DIR = Path.cwd().resolve().parent   # results_lib.py lives in qpp/
if str(QPP_DIR) not in sys.path:
    sys.path.insert(0, str(QPP_DIR))

REGIME = "offline"

res       = pd.read_csv(RESULTS_DIR / f"cold_start_dose_response_{REGIME}_qpp_results.csv", dtype={"seed": str})
legs_meta = json.loads((RESULTS_DIR / f"cold_start_dose_response_{REGIME}_legs_20260902T141545Z.json").read_text())

print("results CSV     :", f"cold_start_dose_response_{REGIME}_qpp_results.csv")
print("n seeds         :", legs_meta["n_seeds"], " seed_base:", legs_meta["seed_base"])
print("checkpoints     :", legs_meta["checkpoints"])
print("models          :", sorted(res["model"].unique()))
print("plan encoder    :", legs_meta.get("plan_encoder"))

In [2]:
# ---- display conventions (kept local so this notebook is standalone) --------
LEG_DISPLAY = {
    "gpt_generated": "DiverSQL (ours)", "tpcds": "TPC-DS", "sqlbarber": "SQLBarber",
    "bootstrapping_lcm": "Bootstrapping-LCM", "sql_factory": "SQL-Factory",
    "e2etune": "E2ETune", "sqlstorm": "SQLStorm",
}
LEG_COLORS = {
    "tpcds": "#9ca3af", "sqlbarber": "#a855f7", "bootstrapping_lcm": "#6366f1",
    "sql_factory": "#0891b2", "e2etune": "#059669", "sqlstorm": "#db2777",
    "gpt_generated": "#ea580c",
}
def lname(l): return LEG_DISPLAY.get(l, l)

LEGS = sorted(res["train_leg"].unique(), key=lambda l: legs_meta["vendi"].get(l, 0))
FULL_N = {l: int(res.loc[res.train_leg == l, "n_queries"].max()) for l in LEGS}
SMALL_N = 5
print("training legs and their full training-set size:")
for l in LEGS:
    print(f"  {lname(l):20s}  n_full = {FULL_N[l]:5d}   Vendi = {legs_meta['vendi'].get(l)}")

training legs and their full training-set size:
  SQLBarber             n_full =   786   Vendi = 16.38
  Bootstrapping-LCM     n_full =   777   Vendi = 18.98
  TPC-DS                n_full =   760   Vendi = 22.72
  SQL-Factory           n_full =   778   Vendi = 52.44
  E2ETune               n_full =   792   Vendi = 55.32
  SQLStorm              n_full =   738   Vendi = 77.64
  DiverSQL (ours)       n_full =   760   Vendi = 106.06


## 1 . Paper Table 5

The per-leg generalization table at each leg's full training set, with the **MSCN** block using
the unbounded z-scored log-runtime head. Same off-diagonal protocol, `results_lib.build_curve`
aggregation and LaTeX generator (one block per model, per-leg raw values, best-per-column
**bold + underline**).


In [ ]:
import results_lib as rl   # same dir: display names, metric directions, build_curve

merged = res[res.model != "mscn_minmax"].copy()   # mscn_minmax is only used for the comparison below
curve_tex = rl.build_curve(merged)
print("merged models :", sorted(curve_tex.model.unique()))


In [ ]:
LEGS_BY_VENDI = sorted(LEGS, key=lambda l: legs_meta["vendi"].get(l, 0))
OURS = "gpt_generated"
TEX_MODELS  = ["mscn", "random_forest", "xgboost", "mlp", "gp_rbf", "lightgbm"]   # wins-notebook order
TEX_METRICS = rl.CORE_METRICS   # log_mae, log_rmse, median_q_error, gmean_q_error, p90_q_error, spearman
TEX_HEADER  = {"log_mae": r"Log\\MAE", "log_rmse": r"Log\\RMSE", "median_q_error": r"Median\\Q-Error",
               "gmean_q_error": r"Mean\\Q-Error", "p90_q_error": r"P90\\Q-Error", "spearman": "Spearman"}
TEX_LEG_ORDER = sorted((l for l in LEGS_BY_VENDI if l != OURS), key=lname) + [OURS]
CAPTION = ("Each model family trained on one dataset, evaluated on the blended held-out set "
           "(offline batch-retrain). The MSCN block uses an unbounded (z-scored log-runtime) "
           "regression head; all other families are unchanged. Leading values per column are bold + underlined.")

# nearest-checkpoint row for (leg, model); the 10**9 sentinel resolves to that leg's own max
def pick_row(curve, leg, model, target_n=10**9):
    sub = curve[(curve.train_leg == leg) & (curve.model == model)].sort_values("n_queries")
    if sub.empty:
        return None
    ns = sub["n_queries"].to_numpy(float)
    return sub.iloc[int(np.argmin(np.abs(ns - float(target_n))))]

def latex_generalization_table(curve, models, leg_order, metrics=TEX_METRICS):
    header = " & ".join([r"\makecell[c]{Model/Dataset}"]
                        + [rf"\makecell[c]{{{TEX_HEADER[m]}}}" for m in metrics])
    lines = [r"\begin{table}[t!]", rf"\caption{{{CAPTION}}}",
             r"\label{tab:qpp-generalization-offline-mscn-unbounded}",
             r"\resizebox{0.75\linewidth}{!}{%", r"\renewcommand{\arraystretch}{0.9}",
             r"\begin{tabular}{l" + "c" * len(metrics) + "}", r"\toprule", header + r" \\", r"\midrule"]
    for mi, model in enumerate(models):
        lines.append(r"\rowcolor{gray!15}")
        lines.append(rf"\multicolumn{{{len(metrics) + 1}}}{{l}}{{\textbf{{{rl.model_display_name(model)}}}}} \\")
        vals = {leg: {m: (r[f"{m}_mean"] if (r := pick_row(curve, leg, model)) is not None else np.nan)
                      for m in metrics} for leg in leg_order}
        best = {}
        for m in metrics:
            col = {leg: vals[leg][m] for leg in leg_order if np.isfinite(vals[leg][m])}
            best[m] = (min(col, key=col.get) if rl.METRIC_DIRECTIONS[m] else max(col, key=col.get)) if col else None
        for leg in leg_order:
            label = "DiverSQL (Ours)" if leg == OURS else rl.display_name(leg)
            cells = [label]
            for m in metrics:
                v = vals[leg][m]
                if not np.isfinite(v):
                    cells.append("--"); continue
                s = f"{v:.3f}"
                if best[m] == leg:
                    s = rf"\underline{{\textbf{{{s}}}}}"
                cells.append(s)
            lines.append(" & ".join(cells) + r" \\")
        lines.append(r"\midrule" if mi < len(models) - 1 else r"\bottomrule")
    lines += [r"\end{tabular}", r"}", r"\end{table}"]
    return "\n".join(lines)

tex = latex_generalization_table(curve_tex, TEX_MODELS, TEX_LEG_ORDER)
tex_path = RESULTS_DIR / "mscn_anomaly_generalization_table_offline_mscn_unbounded.tex"
tex_path.write_text(tex)
print(tex)

In [ ]:
# what the MSCN swap changed: bounded (log_minmax) vs unbounded (log_zscore), per leg, at full n
curve_bounded = rl.build_curve(res[res.model == "mscn_minmax"].assign(model="mscn"))
rows = []
for leg in TEX_LEG_ORDER:
    rb, ru = pick_row(curve_bounded, leg, "mscn"), pick_row(curve_tex, leg, "mscn")
    rows.append(dict(dataset=("DiverSQL (Ours)" if leg == OURS else lname(leg)),
                     logMAE_bounded=rb["log_mae_mean"],   logMAE_unbounded=ru["log_mae_mean"],
                     gmeanQ_bounded=rb["gmean_q_error_mean"], gmeanQ_unbounded=ru["gmean_q_error_mean"],
                     spearman_bounded=rb["spearman_mean"], spearman_unbounded=ru["spearman_mean"]))
print("MSCN block only - bounded (log_minmax) vs unbounded (log_zscore), each leg's full training set:\n")
print(pd.DataFrame(rows).round(3).to_string(index=False))
print("\nDiverSQL stays best-in-column on Log_MAE / GMean-Q / Spearman with the unbounded head;")
print("the classical-model blocks in the table above are identical to the wins-notebook table.")

## 2 . Figure 3 - Log MAE vs. training-set size, per model

`log_mae` plotted directly (lower = better): a curve sloping **down** to the right means
"more training data helps". DiverSQL is the bold orange line. The MSCN panel uses the
`log_zscore` arm; every other panel is the classical model as-run.

In [ ]:
GROWTH_MODELS = ["mscn", "random_forest", "xgboost", "mlp", "gp_rbf", "lightgbm"]
GROWTH_METRIC = "log_mae"
SEED_AGG = "mean"   # across seeds (figure only; Table 5 always uses the mean):
                    #   "mean" | "median"
                    #   "trimmed" : drop best + worst seed, mean of the rest
                    #   "mad"     : drop seeds > 3 scaled MADs from the median, mean of the rest

def _trimmed(x):
    x = np.sort(x.to_numpy())
    return x[1:-1].mean() if len(x) > 2 else x.mean()

def _mad(x, k=3.0):
    x = x.to_numpy(); med = np.median(x)
    mad = 1.4826 * np.median(np.abs(x - med))
    return x.mean() if mad == 0 else x[np.abs(x - med) <= k * mad].mean()

_agg = {"trimmed": _trimmed, "mad": _mad}.get(SEED_AGG, SEED_AGG)

# same as rl.build_curve (average over test legs within each seed first), then
# aggregate the per-seed values with SEED_AGG
_m = merged[~merged["degenerate_fit"].astype(bool)] if "degenerate_fit" in merged else merged
curve_fig = (_m.groupby(["train_leg", "seed", "n_queries", "model"])[rl.METRIC_COLS].mean().reset_index()
               .groupby(["train_leg", "n_queries", "model"])[rl.METRIC_COLS].agg(_agg)
               .add_suffix(f"_{SEED_AGG}").reset_index())

Axis_label_size = 12
ncols = 3
nrows = int(np.ceil(len(GROWTH_MODELS) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(4.4 * ncols, 2.5 * nrows), sharey=False)
axes = np.atleast_1d(axes).flatten()

for i, (ax, model) in enumerate(zip(axes, GROWTH_MODELS)):
    for leg in LEGS_BY_VENDI:
        sub = rl.leg_curve(curve_fig, leg, model)
        if sub.empty:
            continue
        is_ours = leg == OURS
        ax.plot(sub["n_queries"], sub[f"{GROWTH_METRIC}_{SEED_AGG}"], "-o", markersize=4,
                lw=2.5 if is_ours else 1.1, alpha=1.0 if is_ours else 0.5,
                color=rl.LEG_COLORS[leg], label=lname(leg))
    ax.set_xscale("log")

    if i + ncols >= len(GROWTH_MODELS):
        ax.set_xlabel("Training queries", fontsize=Axis_label_size)
    else:
        ax.set_xlabel(""); ax.tick_params(axis="x", labelbottom=False)

    if i % ncols == 0:
        ax.set_ylabel(rl.metric_display_name(GROWTH_METRIC), fontsize=Axis_label_size)
    else:
        ax.set_ylabel("")

    ax.set_title(rl.model_display_name(model), fontsize=Axis_label_size)
    ax.grid(alpha=0.25)
    ax.tick_params(axis='both', labelsize=Axis_label_size)

for ax in axes[len(GROWTH_MODELS):]:
    ax.axis("off")

axes[2].legend(
    fontsize=8.25,
    loc="best",
    labelspacing=0.1,
    handletextpad=0.4,
    handlelength=1.5,
    borderpad=0.3,
    columnspacing=0.8,
    ncol=1,
)

plt.tight_layout()
fig.subplots_adjust(wspace=0.15)

png = RESULTS_DIR / "logmae_growth_grid.png"
fig.savefig(png, dpi=150, bbox_inches="tight")
pdf = RESULTS_DIR / "logmae_growth_grid.pdf"
fig.savefig(pdf, dpi=150, bbox_inches="tight")
plt.show()
print("wrote:", png.name)